In [10]:
import random
import pandas as pd
from datetime import datetime, timedelta

NUM_PATIENTS = 1000
START_DATE = datetime(2026,1,1,8,0)
random.seed(42)

DEPT={
"Patient Arrival":"Reception",
"Registration":"Reception",
"Waiting":"Reception",
"Doctor Consultation":"Clinical",
"Lab Test":"Laboratory",
"Diagnosis":"Clinical",
"Treatment":"Treatment",
"ICU":"ICU",
"Monitoring":"Ward",
"Billing":"Billing",
"Discharged":"Reception"
}

rows=[]
event_id=1

def workflow():
    w=["Patient Arrival","Registration","Waiting","Doctor Consultation"]
    r=random.random()
    if r<0.5:
        w+=["Diagnosis","Treatment","Monitoring","Billing","Discharged"]
    elif r<0.9:
        w+=["Lab Test","Diagnosis","Doctor Consultation","Treatment","Monitoring","Billing","Discharged"]
    else:
        w+=["ICU","Monitoring","Billing","Discharged"]
    return w

for i in range(1,NUM_PATIENTS+1):
    t=START_DATE+timedelta(minutes=random.randint(0,10000))
    case=f"CASE-{i:04d}"
    patient=f"PAT-{i:04d}"
    doctor=f"DOC-{random.randint(101,120)}"
    severity=random.choice(["Low","Medium","High","Critical"])
    visit=random.choice(["OPD","IPD","Emergency"])

    for act in workflow():
        t+=timedelta(minutes=random.randint(5,30))
        rows.append({
            "Event_ID":f"EVT-{event_id:06d}",
            "Case_ID":case,
            "Patient_ID":patient,
            "Visit_Type":visit,
            "Activity":act,
            "Department":DEPT[act],
            "Timestamp":t.strftime("%Y-%m-%d %H:%M:%S"),
            "Doctor_ID":doctor,
            "Severity":severity,
            "Waiting_Time_Minutes":random.randint(5,30),
            "Cost":random.randint(200,5000),
            "Status":"Completed"
        })
        event_id+=1
df=pd.DataFrame(rows)
# ==================================================
# Introduce Data Quality Issues
# ==================================================

# 1. Duplicate Rows (3%)

duplicates = df.sample(300, random_state=1)

df = pd.concat([df, duplicates], ignore_index=True)

# --------------------------------------------------
# 2. NULL Values
# --------------------------------------------------

for column in ["Doctor_ID", "Severity"]:

    idx = df.sample(120, random_state=random.randint(1,1000)).index

    df.loc[idx, column] = None


# --------------------------------------------------
# 3. Inconsistent Activity Names
# --------------------------------------------------

mapping = {

    "Registration": [
        "registration",
        "REGISTRATION"
    ],

    "Doctor Consultation": [
        "Doctor consultation",
        "doctor consultation"
    ],

    "Lab Test": [
        "Lab test",
        "LAB TEST"
    ]

}

for original, variants in mapping.items():

    activity_rows = df[df["Activity"] == original]

    if len(activity_rows) > 0:

        idx = activity_rows.sample(
            min(80, len(activity_rows)),
            random_state=random.randint(1,1000)
        ).index

        for i, row in enumerate(idx):

            df.at[row, "Activity"] = variants[i % len(variants)]


# --------------------------------------------------
# 4. Negative Waiting Time
# --------------------------------------------------

idx = df.sample(80, random_state=9).index

df.loc[idx, "Waiting_Time_Minutes"] = -5


# --------------------------------------------------
# 5. Invalid Status
# --------------------------------------------------

idx = df.sample(60, random_state=10).index

df.loc[idx, "Status"] = "Done"


# --------------------------------------------------
# 6. Blank Timestamp
# --------------------------------------------------

idx = df.sample(50, random_state=11).index

df.loc[idx, "Timestamp"] = ""


# --------------------------------------------------
# 7. Extra Spaces
# --------------------------------------------------

idx = df.sample(100, random_state=12).index

df.loc[idx, "Department"] = (
    " " +
    df.loc[idx, "Department"] +
    " "
)
#-----------------------------------------------------
df.to_csv("hospital_event_log_day5.csv",index=False)

print(df.head())
print(f"Patients: {NUM_PATIENTS}")
print(f"Events: {len(df)}")

display(df)

print(df.isnull().sum())
print("In hospital CSV their are", df.duplicated().sum(), "Duplicate values.")

     Event_ID    Case_ID Patient_ID Visit_Type             Activity  \
0  EVT-000001  CASE-0001   PAT-0001        OPD      Patient Arrival   
1  EVT-000002  CASE-0001   PAT-0001        OPD         Registration   
2  EVT-000003  CASE-0001   PAT-0001        OPD              Waiting   
3  EVT-000004  CASE-0001   PAT-0001        OPD  Doctor Consultation   
4  EVT-000005  CASE-0001   PAT-0001        OPD            Diagnosis   

  Department            Timestamp Doctor_ID Severity  Waiting_Time_Minutes  \
0  Reception  2026-01-02 14:52:00   DOC-101     High                     8   
1  Reception  2026-01-02 14:59:00   DOC-101     High                    23   
2  Reception  2026-01-02 15:05:00   DOC-101     High                     5   
3   Clinical  2026-01-02 15:16:00   DOC-101     High                    12   
4   Clinical  2026-01-02 15:40:00   DOC-101     High                     5   

   Cost     Status  
0  4667  Completed  
1  3656  Completed  
2   967  Completed  
3  4339  Completed  

,Event_ID,Case_ID,Patient_ID,Visit_Type,Activity,Department,Timestamp,Doctor_ID,Severity,Waiting_Time_Minutes,Cost,Status
0,EVT-000001,CASE-0001,PAT-0001,OPD,Patient Arrival,Reception,2026-01-02 14:52:00,DOC-101,High,8,4667,Completed
1,EVT-000002,CASE-0001,PAT-0001,OPD,Registration,Reception,2026-01-02 14:59:00,DOC-101,High,23,3656,Completed
2,EVT-000003,CASE-0001,PAT-0001,OPD,Waiting,Reception,2026-01-02 15:05:00,DOC-101,High,5,967,Completed
3,EVT-000004,CASE-0001,PAT-0001,OPD,Doctor Consultation,Clinical,2026-01-02 15:16:00,DOC-101,High,12,4339,Completed
4,EVT-000005,CASE-0001,PAT-0001,OPD,Diagnosis,Clinical,2026-01-02 15:40:00,DOC-101,High,5,4797,Completed
...,...,...,...,...,...,...,...,...,...,...,...,...
10008,EVT-009241,CASE-0951,PAT-0951,OPD,Treatment,Treatment,2026-01-06 16:00:00,DOC-101,Medium,17,3847,Completed
10009,EVT-002858,CASE-0295,PAT-0295,Emergency,Doctor Consultation,Clinical,2026-01-08 04:39:00,DOC-111,Low,17,4608,Completed
10010,EVT-007523,CASE-0774,PAT-0774,OPD,Discharged,Reception,2026-01-08 06:25:00,DOC-113,Medium,16,3040,Completed
10011,EVT-004425,CASE-0454,PAT-0454,Emergency,Monitoring,Ward,2026-01-07 06:41:00,DOC-118,Medium,9,2450,Completed


Event_ID                  0
Case_ID                   0
Patient_ID                0
Visit_Type                0
Activity                  0
Department                0
Timestamp                 0
Doctor_ID               120
Severity                120
Waiting_Time_Minutes      0
Cost                      0
Status                    0
dtype: int64
In hospital CSV their are 262 Duplicate values.
